In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Define paths
ARTIFACT_DIR = Path('./artifacts')
BASELINE_STATS = ARTIFACT_DIR / 'baseline_stats.json'
DETECTED_ANOMALIES_OUT = ARTIFACT_DIR / 'detected_anomalies.json'

# Anomaly thresholds
Z_SCORE_THRESHOLD = 3.0
IQR_MULTIPLIER = 1.5

## Step 1: Load Baseline Statistics

In [ ]:
# Load baseline stats
if not BASELINE_STATS.exists():
    raise FileNotFoundError(f"Run Notebook 1 first to generate: {BASELINE_STATS}")

with open(BASELINE_STATS, 'r') as f:
    baseline_stats = json.load(f)

print(f"Loaded baseline statistics for {len(baseline_stats)} metrics")

## Step 2: Load New Run Metrics

In [ ]:
# Load new run from Spark table
from pyspark.sql import functions as F

# Define metrics table
METRICS_TABLE = "bms_ds_prod.bms_ds_dasc.temp_raw_metrics"

# Read the most recent run from the metrics table
sdf = spark.table(METRICS_TABLE).select(
    F.col('date'),
    F.col('metric_name'),
    F.col('metric_value')
)

# Get the most recent date (new run)
latest_date = sdf.select(F.max('date')).collect()[0][0]
print(f"Loading new run from date: {latest_date}")

# Filter for the latest date and pivot
new_run_sdf = (
    sdf.filter(F.col('date') == latest_date)
    .withColumn('metric_value', F.col('metric_value').cast('double'))
    .groupBy('date')
    .pivot('metric_name')
    .agg(F.first('metric_value'))
)

# Convert to pandas and then to dict
new_run_df = new_run_sdf.toPandas()

# Drop date column and convert first row to dict
if 'date' in new_run_df.columns:
    new_run_df = new_run_df.drop(columns=['date'])

new_run = new_run_df.iloc[0].to_dict()
print(f"Loaded new run with {len(new_run)} metrics")

## Step 3: Detect Anomalies

In [ ]:
def detect_anomalies(new_run, baseline_stats, z_thresh=3.0, iqr_multiplier=1.5):
    """
    Detect anomalies using z-score and IQR-based tests.
    
    Args:
        new_run: Dict mapping metric_name -> value
        baseline_stats: Dict mapping metric_name -> stats dict
        z_thresh: Z-score threshold for anomaly detection
        iqr_multiplier: IQR multiplier for outlier detection
        
    Returns:
        Dict mapping anomalous_metric -> anomaly details
    """
    anomalies = {}
    
    for metric, value in new_run.items():
        # Skip non-numeric values
        try:
            v = float(value)
        except (ValueError, TypeError):
            anomalies[metric] = {
                'value': value,
                'reason': 'non_numeric',
                'z_score': None,
                'iqr_flag': False,
                'z_flag': False
            }
            continue
        
        # Check if baseline exists
        stats = baseline_stats.get(metric)
        if not stats or stats.get('n', 0) == 0:
            anomalies[metric] = {
                'value': v,
                'reason': 'no_baseline',
                'z_score': None,
                'iqr_flag': False,
                'z_flag': False
            }
            continue
        
        # Extract baseline statistics
        mean = stats.get('mean')
        std = stats.get('std') or 0.0
        q1 = stats.get('q1')
        q3 = stats.get('q3')
        iqr = stats.get('IQR') or 0.0
        
        # Compute z-score
        z_score = None
        z_flag = False
        if std > 0:
            z_score = (v - mean) / std
            z_flag = abs(z_score) > z_thresh
        
        # Compute IQR-based outlier flag
        iqr_flag = False
        lower_bound = None
        upper_bound = None
        if iqr > 0:
            lower_bound = q1 - iqr_multiplier * iqr
            upper_bound = q3 + iqr_multiplier * iqr
            iqr_flag = (v < lower_bound) or (v > upper_bound)
        
        # Flag anomaly if either test triggers
        if z_flag or iqr_flag:
            anomalies[metric] = {
                'value': float(v),
                'z_score': float(z_score) if z_score is not None else None,
                'z_flag': bool(z_flag),
                'iqr_flag': bool(iqr_flag),
                'lower_iqr': float(lower_bound) if lower_bound is not None else None,
                'upper_iqr': float(upper_bound) if upper_bound is not None else None,
                'baseline_mean': float(mean),
                'baseline_std': float(std)
            }
    
    return anomalies

# Run anomaly detection
detected_anomalies = detect_anomalies(
    new_run, 
    baseline_stats, 
    z_thresh=Z_SCORE_THRESHOLD, 
    iqr_multiplier=IQR_MULTIPLIER
)

print(f"\n{'='*60}")
print(f"Detected {len(detected_anomalies)} anomalous metrics")
print(f"{'='*60}")

# Display detected anomalies
for metric, details in list(detected_anomalies.items())[:10]:
    print(f"\n{metric}:")
    print(f"  Value: {details.get('value')}")
    print(f"  Z-score: {details.get('z_score')}")
    print(f"  Z-flag: {details.get('z_flag')}, IQR-flag: {details.get('iqr_flag')}")

## Step 4: Save Detected Anomalies

In [ ]:
# Save anomalies
with open(DETECTED_ANOMALIES_OUT, 'w') as f:
    json.dump(detected_anomalies, f, indent=2)

print(f"\n✓ Saved detected anomalies to: {DETECTED_ANOMALIES_OUT}")
print("\n" + "="*60)
print("✓ Notebook 2 Complete — Anomalies Detected")
print("="*60)